In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W

In [0]:
catalog = "proyecto"
schema_silver = "silver"
schema_gold = "golden"
gold = "abfss://golden@adlscshm.dfs.core.windows.net/instacart/"

In [0]:
orders  = spark.table(f"{catalog}.{schema_silver}.orders")
ops     = spark.table(f"{catalog}.{schema_silver}.order_products")
prods   = spark.table(f"{catalog}.{schema_silver}.products_enriched")


In [0]:
fact = (
    ops.alias("op")
    .join(orders.alias("o"), "cod_order", "inner")
    .join(prods.alias("p"), "cod_product", "left")
    .select(
        "cod_order", "cod_user",
        "cod_product", "des_product_name",
        "cod_aisle", "des_aisle",
        "cod_department", "des_department",
        "val_add_to_cart_order", "val_reordered",
        "des_eval_set", "val_order_number",
        "val_order_dow", "val_order_hour_of_day",
        "val_days_since_prior_order",
        "flg_weekend", "cat_daypart"
    )
).dropDuplicates()

In [0]:
first_buys = (
    fact.select("cod_user","cod_product","val_order_number")
        .withColumn("rn", F.row_number().over(W.partitionBy("cod_user","cod_product").orderBy("val_order_number")))
        .where(F.col("rn")==1)
        .groupBy("cod_product").agg(F.count("*").alias("val_first_time_buys"))
)

metrics_product = (
    fact.groupBy("cod_product","des_product_name","cod_department","des_department","cod_aisle","des_aisle")
        .agg(
            F.count("*").alias("val_items"),
            F.countDistinct("cod_order").alias("val_orders"),
            F.countDistinct("cod_user").alias("val_users"),
            F.avg("val_reordered").alias("val_reorder_rate"),
            F.avg(F.when(F.col("val_add_to_cart_order")<=5,1).otherwise(0)).alias("val_top_cart_ratio"),
            F.expr("percentile_approx(val_add_to_cart_order, 0.5)").alias("val_add_to_cart_median")
        )
        .join(first_buys, "cod_product", "left")
        .fillna(0, subset=["val_first_time_buys"])
)

metrics_time = (
    fact.groupBy("val_order_dow","cat_daypart")
        .agg(
            F.countDistinct("cod_order").alias("val_orders"),
            F.count("*").alias("val_items"),
            F.countDistinct("cod_user").alias("val_users"),
            F.avg("val_reordered").alias("val_reorder_rate")
        )
)

In [0]:
fact.write.format("delta").mode("overwrite").option("overwriteSchema","true")\
    .option("path", f"{gold}fact_order_items")\
    .saveAsTable(f"{catalog}.{schema_gold}.fact_order_items")

metrics_product.write.format("delta").mode("overwrite").option("overwriteSchema","true")\
    .option("path", f"{gold}metrics_product")\
    .saveAsTable(f"{catalog}.{schema_gold}.metrics_product")

metrics_time.write.format("delta").mode("overwrite").option("overwriteSchema","true")\
    .option("path", f"{gold}metrics_time")\
    .saveAsTable(f"{catalog}.{schema_gold}.metrics_time")

In [0]:
%sql 
select * 
from proyecto.golden.metrics_time